# Evaluasi Metrik dan Toleransi (Eq 2)
Notebook untuk mensimulasikan keseimbangan grafik sesuai paparan Paper (Table II).

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from geevo.evolution.generator import EvolutionaryGraphGeneration
from geevo.evolution.balancer import Balancer
from geevo import nodes as n
from geevo.graph import Graph
from geevo.agent_simulation import Agent
import warnings
warnings.filterwarnings('ignore')

# Seed untuk reproducibility
random.seed(42)
np.random.seed(42)


def generate_graph_from_conf(conf):
    egg = EvolutionaryGraphGeneration(conf)
    if egg.run(steps=50):
        return Graph(conf, egg.get_best())
    egg.run(steps=50)
    return Graph(conf, egg.get_best())


def run_evaluation(graphs, alphas):
    """Evaluasi balancer pada kumpulan graf untuk setiap alpha.
    Output DataFrame mencakup kolom Fitness Std Dev untuk mengukur stabilitas.
    """
    results = []
    agents = [Agent(behavior='aggressive'), Agent(behavior='passive'), Agent(behavior='random')]
    total = len(graphs)

    for alpha in alphas:
        metrics = {
            "Total Runs": 0, "Initial Balanced": 0, "Balanced": 0,
            "Improved": 0, "Execution Times": [], "Generations": [],
            "Final Fitnesses": []
        }

        for idx, g in enumerate(graphs):
            print(f"  alpha={alpha} | graph {idx+1}/{total}...", end="\r")
            pools = g.get_nodes_of(n.Pool)
            if not pools:
                continue

            pool_id = random.randint(0, len(pools) - 1)
            t = random.randint(10, 30)
            x = random.randint(20, 100)

            metrics["Total Runs"] += 1

            b = Balancer(
                agent=agents, config=g.config, edge_list=g.edge_list,
                balance_pool_ids=[pool_id], n_sim_steps=t, balance_value=x,
                alpha=alpha, fitness_func="obj4", pop_size=20, n_sim=10
            )

            initial_fitness = b.get_ind_fitness(b.init_ind())
            if initial_fitness >= b.threshold:
                metrics["Initial Balanced"] += 1

            start_time = time.time()
            final_fitness, iterations = b.run(steps=500)
            end_time = time.time()

            if final_fitness is None:
                final_fitness = b.get_ind_fitness(b.population[0])

            metrics["Execution Times"].append(end_time - start_time)
            metrics["Generations"].append(iterations)
            metrics["Final Fitnesses"].append(final_fitness)

            if final_fitness >= b.threshold:
                metrics["Balanced"] += 1
            if final_fitness > initial_fitness:
                metrics["Improved"] += 1

        print(f"  alpha={alpha} | selesai ({metrics['Total Runs']} runs)          ")
        total_runs = metrics["Total Runs"]
        if total_runs > 0:
            res = {
                "Alpha (\u03b1)": alpha,
                "Balanced %": f"{round((metrics['Balanced'] / total_runs) * 100, 2)}%",
                "Initial Balanced %": f"{round((metrics['Initial Balanced'] / total_runs) * 100, 2)}%",
                "Improved %": f"{round((metrics['Improved'] / total_runs) * 100, 2)}%",
                "Median Generation": np.median(metrics["Generations"]),
                "Median Execution Time (s)": round(np.median(metrics["Execution Times"]), 3),
                "Fitness Std Dev": round(float(np.std(metrics["Final Fitnesses"])), 4)
            }
            results.append(res)

    return pd.DataFrame(results)


def run_evaluation_per_agent(graphs, alphas):
    """Evaluasi terpisah per tipe agen (aggressive, passive, random).
    Output DataFrame dengan kolom tambahan 'Agent'.
    """
    results = []
    agent_types = ['aggressive', 'passive', 'random']
    total = len(graphs)

    for alpha in alphas:
        for agent_name in agent_types:
            agent = Agent(behavior=agent_name)
            metrics = {
                "Total Runs": 0, "Balanced": 0, "Improved": 0,
                "Execution Times": [], "Generations": []
            }

            for idx, g in enumerate(graphs):
                print(f"  alpha={alpha} | agent={agent_name} | graph {idx+1}/{total}...", end="\r")
                pools = g.get_nodes_of(n.Pool)
                if not pools:
                    continue

                pool_id = random.randint(0, len(pools) - 1)
                t = random.randint(10, 30)
                x = random.randint(20, 100)

                metrics["Total Runs"] += 1

                b = Balancer(
                    agent=[agent], config=g.config, edge_list=g.edge_list,
                    balance_pool_ids=[pool_id], n_sim_steps=t, balance_value=x,
                    alpha=alpha, fitness_func="obj4", pop_size=20, n_sim=10
                )

                initial_fitness = b.get_ind_fitness(b.init_ind())

                start_time = time.time()
                final_fitness, iterations = b.run(steps=500)
                end_time = time.time()

                if final_fitness is None:
                    final_fitness = b.get_ind_fitness(b.population[0])

                metrics["Execution Times"].append(end_time - start_time)
                metrics["Generations"].append(iterations)

                if final_fitness >= b.threshold:
                    metrics["Balanced"] += 1
                if final_fitness > initial_fitness:
                    metrics["Improved"] += 1

            print(f"  alpha={alpha} | agent={agent_name} | selesai ({metrics['Total Runs']} runs)          ")
            total_runs = metrics["Total Runs"]
            if total_runs > 0:
                res = {
                    "Alpha (\u03b1)": alpha,
                    "Agent": agent_name,
                    "Balanced %": f"{round((metrics['Balanced'] / total_runs) * 100, 2)}%",
                    "Improved %": f"{round((metrics['Improved'] / total_runs) * 100, 2)}%",
                    "Median Generation": np.median(metrics["Generations"]),
                    "Median Exec Time (s)": round(np.median(metrics["Execution Times"]), 3)
                }
                results.append(res)

    return pd.DataFrame(results)

In [ ]:
print("Mempersiapkan Dataset (Abstract EGG + GDD)...\n")

# 1. Abstract Dataset — 30 graf untuk sample size yang representatif
base_conf = {n.Source: 3, n.RandomGate: 2, n.Pool: 4, n.Converter: 1}
abstract_graphs = []
for i in range(30):
    print(f"  Generating abstract graph {i+1}/30...", end="\r")
    abstract_graphs.append(generate_graph_from_conf(base_conf))
print(f"  Abstract graphs siap: {len(abstract_graphs)} graf          ")

# 2. GDD MMORPG — lebih kompleks
mmorpg_conf = {n.Source: 2, n.RandomGate: 3, n.Pool: 5, n.Converter: 3}
mmorpg_graphs = []
for i in range(10):
    print(f"  Generating MMORPG graph {i+1}/10...", end="\r")
    mmorpg_graphs.append(generate_graph_from_conf(mmorpg_conf))
print(f"  MMORPG graphs siap: {len(mmorpg_graphs)} graf          ")

# 3. GDD VHS/Horror — lebih sederhana
vhs_conf = {n.Source: 1, n.RandomGate: 2, n.Pool: 3, n.Converter: 1}
vhs_graphs = []
for i in range(10):
    print(f"  Generating VHS/Horror graph {i+1}/10...", end="\r")
    vhs_graphs.append(generate_graph_from_conf(vhs_conf))
print(f"  VHS/Horror graphs siap: {len(vhs_graphs)} graf          ")

alphas = [0.05, 0.01, 0.0]
print("\nDataset siap. Total:", len(abstract_graphs), "abstract +", len(mmorpg_graphs), "MMORPG +", len(vhs_graphs), "VHS")

In [ ]:
print("[1] Evaluasi Dataset Generator Abstrak GEEvo (n=30)")
df_abstract = run_evaluation(abstract_graphs, alphas)
display(df_abstract)

In [ ]:
print("[2] Evaluasi Dataset GDD — MMORPG (n=10)")
df_gdd_mmorpg = run_evaluation(mmorpg_graphs, alphas)
display(df_gdd_mmorpg)

In [ ]:
print("[3] Evaluasi Dataset GDD — VHS/Horror (n=10)")
df_gdd_vhs = run_evaluation(vhs_graphs, alphas)
display(df_gdd_vhs)

In [ ]:
print("[4] Evaluasi Breakdown per Tipe Agen — Abstract (n=30)")
df_per_agent = run_evaluation_per_agent(abstract_graphs, alphas)
display(df_per_agent)

In [ ]:
print("[5] Plot Konvergensi Fitness — Contoh MMORPG graf pertama, alpha=0.05")

g_sample = mmorpg_graphs[0]
pools_sample = g_sample.get_nodes_of(n.Pool)
pool_id_sample = random.randint(0, len(pools_sample) - 1)

b_sample = Balancer(
    agent=[Agent(behavior='aggressive'), Agent(behavior='passive'), Agent(behavior='random')],
    config=g_sample.config, edge_list=g_sample.edge_list,
    balance_pool_ids=[pool_id_sample], n_sim_steps=20, balance_value=50,
    alpha=0.05, fitness_func="obj4", pop_size=20, n_sim=10
)

b_sample.run(steps=500)

fig, ax = plt.subplots(figsize=(10, 5))
generations = list(range(len(b_sample.monitor["best"])))
ax.plot(generations, b_sample.monitor["best"], label="Best Fitness", linewidth=2)
ax.plot(generations, b_sample.monitor["avg"], label="Avg Fitness", linewidth=2, linestyle='--')
ax.axhline(y=0.95, color='red', linestyle=':', linewidth=1.5, label="Threshold (\u03b1=0.05)")
ax.set_xlabel("Generasi")
ax.set_ylabel("Fitness")
ax.set_title("Konvergensi Fitness — Graf MMORPG, \u03b1=0.05")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Tabel Perbandingan dengan Paper GEEvo (Table II)
Perbandingan eksplisit antara hasil TA dan referensi Rupp & Eckert (2024).

In [ ]:
def _get_val(df, alpha, col):
    """Ambil nilai dari DataFrame hasil evaluasi untuk alpha dan kolom tertentu."""
    row = df[df["Alpha (\u03b1)"] == alpha]
    if row.empty:
        return "N/A"
    return row.iloc[0][col]


comparison_data = {
    "Sumber": [
        "Paper GEEvo (n=194)",
        "TA — Abstract (n=30)",
        "TA — MMORPG (n=10)",
        "TA — VHS/Horror (n=10)"
    ],
    "\u03b1=0.05 Balanced %": [
        "93.3%",
        _get_val(df_abstract, 0.05, "Balanced %"),
        _get_val(df_gdd_mmorpg, 0.05, "Balanced %"),
        _get_val(df_gdd_vhs, 0.05, "Balanced %")
    ],
    "\u03b1=0.01 Balanced %": [
        "83.0%",
        _get_val(df_abstract, 0.01, "Balanced %"),
        _get_val(df_gdd_mmorpg, 0.01, "Balanced %"),
        _get_val(df_gdd_vhs, 0.01, "Balanced %")
    ],
    "\u03b1=0.05 Median Gen": [
        1,
        _get_val(df_abstract, 0.05, "Median Generation"),
        _get_val(df_gdd_mmorpg, 0.05, "Median Generation"),
        _get_val(df_gdd_vhs, 0.05, "Median Generation")
    ],
    "\u03b1=0.05 Exec Time (s)": [
        18.4,
        _get_val(df_abstract, 0.05, "Median Execution Time (s)"),
        _get_val(df_gdd_mmorpg, 0.05, "Median Execution Time (s)"),
        _get_val(df_gdd_vhs, 0.05, "Median Execution Time (s)")
    ],
    "\u03b1=0.05 Fitness Std Dev": [
        "N/A",
        _get_val(df_abstract, 0.05, "Fitness Std Dev"),
        _get_val(df_gdd_mmorpg, 0.05, "Fitness Std Dev"),
        _get_val(df_gdd_vhs, 0.05, "Fitness Std Dev")
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("[6] Tabel Perbandingan Hasil TA vs Paper GEEvo")
display(df_comparison)

In [ ]:
import os
import networkx as nx
import matplotlib.pyplot as plt

# Load file GML
file_name = 'coc_machinations.gml'

try:
    G = nx.read_gml(file_name)
    print(f"\u2705 Berhasil memuat: {G.number_of_nodes()} Node & {G.number_of_edges()} Edges")

    # Visualisasi Graf
    plt.figure(figsize=(15, 10))
    pos = nx.spring_layout(G, k=0.3, iterations=50, seed=42)

    nx.draw(G, pos,
            with_labels=True,
            node_color='skyblue',
            node_size=1500,
            edge_color='#999999',
            font_size=10,
            font_weight='bold',
            arrowsize=20)

    plt.title("Visualisasi Diagram Machinations CoC", fontsize=15)
    plt.show()

    # Analisis Sederhana: Node paling penting (Degree Centrality)
    print("\n--- Analisis Struktur ---")
    centrality = nx.degree_centrality(G)
    sorted_centrality = sorted(centrality.items(), key=lambda x: x[1], reverse=True)

    print("5 Node dengan koneksi terbanyak:")
    for node, score in sorted_centrality[:5]:
        print(f"Node '{node}': {score:.2f}")

except FileNotFoundError:
    print("\u274c Error: File GML tidak ditemukan. Pastikan nama file sudah benar.")
except Exception as e:
    print(f"\u274c Terjadi kesalahan: {e}")

In [ ]:
import re

G_geevo = G.copy()

def infer_geevo_type(name: str) -> str:
    s = str(name).lower()
    if any(k in s for k in ['source', 'mine', 'produksi']):
        return 'Source'
    if any(k in s for k in ['gate', 'drain', 'collect']):
        return 'RandomGate'
    if any(k in s for k in ['inventory', 'capacity', 'level', 'storage']):
        return 'Pool'
    return 'Converter'

# 1) Hapus self-loop
G_geevo.remove_edges_from(nx.selfloop_edges(G_geevo))

# 2) Hapus node utilitas/auxiliary dan bypass alirannya
aux_pattern = re.compile(r'\b(list|cost|duration)\b', re.I)
aux_nodes = [nd for nd in list(G_geevo.nodes) if aux_pattern.search(str(nd))]

for node in aux_nodes:
    if node not in G_geevo:
        continue
    preds = list(G_geevo.predecessors(node))
    succs = list(G_geevo.successors(node))
    for u in preds:
        for v in succs:
            if u != v:
                G_geevo.add_edge(u, v)
    G_geevo.remove_node(node)

# 3) Simpan komponen terhubung lemah terbesar
if G_geevo.number_of_nodes() > 0 and not nx.is_weakly_connected(G_geevo):
    lcc = max(nx.weakly_connected_components(G_geevo), key=len)
    G_geevo = G_geevo.subgraph(lcc).copy()

# 4) Putus siklus agar lebih stabil untuk simulasi Geevo
while True:
    try:
        cyc = nx.find_cycle(G_geevo, orientation='original')
        cycle_edges = [(u, v) for u, v, _ in cyc]

        deg_cent = nx.degree_centrality(G_geevo)
        edge_to_remove = min(
            cycle_edges,
            key=lambda e: (
                deg_cent.get(e[1], 0),
                G_geevo.in_degree(e[1]) + G_geevo.out_degree(e[1])
            )
        )
        G_geevo.remove_edge(*edge_to_remove)
    except nx.NetworkXNoCycle:
        break

# 5) Hapus isolate yang tersisa
G_geevo.remove_nodes_from(list(nx.isolates(G_geevo)))

# 6) Tambahkan atribut tipe node Geevo
for node in G_geevo.nodes:
    G_geevo.nodes[node]['geevo_type'] = infer_geevo_type(node)

# 7) Ringkasan hasil
deg_cent = nx.degree_centrality(G_geevo)
sorted_deg = sorted(deg_cent.items(), key=lambda x: x[1], reverse=True)

print(f"Nodes: {G_geevo.number_of_nodes()}")
print(f"Edges: {G_geevo.number_of_edges()}")
print(f"Cycles: {0 if nx.is_directed_acyclic_graph(G_geevo) else 'Masih ada siklus'}")
print("\nTop 10 node paling penting:")
for node, score in sorted_deg[:10]:
    print(f"- {node}: {score:.3f} | {G_geevo.nodes[node]['geevo_type']}")

# 8) Visualisasi graf yang sudah disempurnakan
plt.figure(figsize=(18, 12))
pos_geevo = nx.spring_layout(G_geevo, seed=42, k=0.5)

node_sizes = [900 + 6000 * deg_cent.get(nd, 0) for nd in G_geevo.nodes()]
node_colors = [deg_cent.get(nd, 0) for nd in G_geevo.nodes()]

edges = nx.draw_networkx_edges(
    G_geevo, pos_geevo,
    arrowstyle='-|>', arrowsize=16,
    edge_color='gray', alpha=0.35
)
nodes = nx.draw_networkx_nodes(
    G_geevo, pos_geevo,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.viridis
)
nx.draw_networkx_labels(G_geevo, pos_geevo, font_size=9)

plt.colorbar(nodes, label='Degree Centrality')
plt.title("Graf yang disempurnakan untuk constraint Geevo", fontsize=15)
plt.axis('off')
plt.show()

In [ ]:
import os

# Buat folder output
os.makedirs("output_results", exist_ok=True)

# Simpan semua DataFrame ke CSV
df_abstract.to_csv("output_results/df_abstract.csv", index=False)
df_gdd_mmorpg.to_csv("output_results/df_gdd_mmorpg.csv", index=False)
df_gdd_vhs.to_csv("output_results/df_gdd_vhs.csv", index=False)
df_per_agent.to_csv("output_results/df_per_agent.csv", index=False)
df_comparison.to_csv("output_results/df_comparison.csv", index=False)

# Simpan plot konvergensi
fig_conv, ax_conv = plt.subplots(figsize=(10, 5))
generations = list(range(len(b_sample.monitor["best"])))
ax_conv.plot(generations, b_sample.monitor["best"], label="Best Fitness", linewidth=2)
ax_conv.plot(generations, b_sample.monitor["avg"], label="Avg Fitness", linewidth=2, linestyle='--')
ax_conv.axhline(y=0.95, color='red', linestyle=':', linewidth=1.5, label="Threshold (α=0.05)")
ax_conv.set_xlabel("Generasi")
ax_conv.set_ylabel("Fitness")
ax_conv.set_title("Konvergensi Fitness — Graf MMORPG, α=0.05")
ax_conv.legend()
ax_conv.grid(True, alpha=0.3)
plt.tight_layout()
fig_conv.savefig("output_results/plot_konvergensi_mmorpg.png", dpi=150, bbox_inches='tight')
plt.close(fig_conv)

print("✅ Semua hasil tersimpan di folder output_results/")
print("   - df_abstract.csv")
print("   - df_gdd_mmorpg.csv")
print("   - df_gdd_vhs.csv")
print("   - df_per_agent.csv")
print("   - df_comparison.csv")
print("   - plot_konvergensi_mmorpg.png")
